# Pneumonia Localization

In [ ]:
from pathlib import Path
import json
import subprocess
import sys
import pandas as pd

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "scripts" / "run_project.py").is_file():
    PROJECT_ROOT = PROJECT_ROOT.parent
SCRIPT = PROJECT_ROOT / "scripts" / "run_project.py"
assert SCRIPT.is_file(), f"Project runner not found: {SCRIPT}"

def run_stage(flag):
    subprocess.run([sys.executable, str(SCRIPT), flag], cwd=PROJECT_ROOT, check=True)

print("Project:", PROJECT_ROOT)
print("COCO source:", PROJECT_ROOT / "train" / "_annotations.coco.json")

## 1. Parse COCO, validate boxes, and make the stratified image-level split

In [ ]:
run_stage("--prepare-data")
quality = json.loads((PROJECT_ROOT / "data" / "splits" / "data_quality_report.json").read_text())
splits = {name: pd.read_csv(PROJECT_ROOT / "data" / "splits" / f"{name}.csv") for name in ("train", "val", "test")}
print(json.dumps(quality, indent=2))
print("Split sizes:", {name: len(frame) for name, frame in splits.items()})
print("Split class counts:\n", pd.concat([frame.assign(split=name) for name, frame in splits.items()]).groupby(["split", "class_name"]).size())
assert sum(map(len, splits.values())) == quality["image_count"]

## 2. Train and evaluate classical baselines


In [ ]:
run_stage("--train-baselines")
baseline_results = json.loads((PROJECT_ROOT / "results" / "metrics" / "baselines.json").read_text())
pd.DataFrame({name: {"validation_accuracy": result["validation"]["accuracy"], "test_accuracy": result["test"]["accuracy"]} for name, result in baseline_results.items()}).T

## 3. Preprocess using training-only normalization and train the CAM-compatible CNN

In [ ]:
run_stage("--train-cnn")
normalization = json.loads((PROJECT_ROOT / "results" / "metrics" / "normalization.json").read_text())
history = json.loads((PROJECT_ROOT / "results" / "metrics" / "cnn_training_history.json").read_text())
print("Training-split normalization:", normalization)
pd.DataFrame(history)

## 4. Generate validation CAMs, apply DFS, and select the threshold

In [ ]:
run_stage("--select-cam-threshold")
threshold_results = json.loads((PROJECT_ROOT / "results" / "metrics" / "selected_cam_threshold.json").read_text())
print(json.dumps(threshold_results, indent=2))

## 5. Train Faster R-CNN with validation checkpoint selection

In [ ]:
run_stage("--train-rcnn")
rcnn_history = json.loads((PROJECT_ROOT / "results" / "metrics" / "rcnn_training_history.json").read_text())
pd.DataFrame(rcnn_history)

## 6. Final held-out test evaluation and CAM/Faster R-CNN comparison


In [ ]:
run_stage("--evaluate")
run_stage("--generate-results")
cam_metrics = json.loads((PROJECT_ROOT / "results" / "metrics" / "test_cam.json").read_text())["metrics"]
rcnn_metrics = json.loads((PROJECT_ROOT / "results" / "metrics" / "test_rcnn.json").read_text())["metrics"]
baseline_metrics = json.loads((PROJECT_ROOT / "results" / "metrics" / "baseline_test_metrics.json").read_text())
comparison = pd.DataFrame({"CAM": cam_metrics, "Faster R-CNN": rcnn_metrics})
display(comparison)
display(pd.DataFrame({name: values for name, values in baseline_metrics.items()}).T)
print("Per-image CAM table:", PROJECT_ROOT / "results" / "tables" / "test_cam_per_image.csv")
print("Per-image Faster R-CNN table:", PROJECT_ROOT / "results" / "tables" / "test_rcnn_per_image.csv")

## 7. Results generated by this run


In [ ]:
final_results = json.loads((PROJECT_ROOT / 'results' / 'metrics' / 'final_results.json').read_text())
display(pd.DataFrame(final_results['classification_and_localization']))
print('CAM positive-only mean IoU:', final_results['cam_positive_only_iou']['mean_iou'])
print('Faster R-CNN positive-only mean IoU:', final_results['rcnn_positive_only_iou']['mean_iou'])
print('Selected CAM threshold:', final_results['cam_threshold'])